# 03 · Credit Risk Classification — LR vs RF vs XGBoost

**Project 8 / 10**  
Three-model credit default classification. AUC-ROC, Gini, Precision-Recall.  
Feature importance and FICO-score sensitivity analysis.

> Author: Niraj Neupane | github.com/nirajneupane17

In [ ]:
import sys; sys.path.insert(0,"../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from feature_engineering import build_credit_features, add_credit_interactions
from ml_models import CreditRiskClassifier
print("Libraries loaded")

In [ ]:
cred = pd.read_csv("../data/credit_data.csv")
print(f"Shape       : {cred.shape}")
print(f"Default rate: {cred['default'].mean()*100:.1f}%")
cred.describe()

In [ ]:
cred_e = add_credit_interactions(cred)
Xc_cols = ["fico_score","dti_ratio","loan_to_value","emp_years",
           "num_accounts","delinquencies","credit_history",
           "debt_burden","credit_score","income_ratio"]
Xc_cols = [c for c in Xc_cols if c in cred_e.columns]
Xc = cred_e[Xc_cols]; yc = cred_e["default"]
Xct,Xce,yct,yce = train_test_split(Xc,yc,test_size=0.2,random_state=42,stratify=yc)
print(f"Train: {Xct.shape}  Test: {Xce.shape}")

In [ ]:
clf = CreditRiskClassifier()
clf.fit(Xct, yct)
report = clf.evaluate_all(Xce, yce)
report

In [ ]:
img = plt.imread("../results/03_credit_roc_auc.png")
fig, ax = plt.subplots(figsize=(16,6)); ax.imshow(img); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Feature importance
import pandas as pd
fi_rf  = pd.Series(clf.rf.feature_importances_, index=Xct.columns).sort_values(ascending=False)
fi_xgb = pd.Series(clf.xg.feature_importances_, index=Xct.columns).sort_values(ascending=False)
print("Top 5 RF  features:"); print(fi_rf.head(5).to_string())
print("\nTop 5 XGB features:"); print(fi_xgb.head(5).to_string())
print(f"\nFeature importance correlation (RF vs XGB): {fi_rf.corr(fi_xgb):.3f}")